# ECE 311 Lab Final:

## Due Date: **Tuesday, 5/12 @ 11:59PM**
## Late Submission: Wednesday, 5/13 @ 5:00AM
### Note: a **10% penalty will be applied for each hour** your submission is late until the late submission time!

This lab final will review the import concepts from the course. Much of this lab should be familiar from previous labs. We encourage you to look back on your previous labs to remind you how to produce your results and evaluate their correctness. Enough talking, let's get started!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from skimage.io import imread
from scipy import signal
from pz_plot import pz_plot
from scipy.io import wavfile
from numpy.random import randn
from IPython.display import Audio

#Utility function for dB scaling of magnitude spectra
def sig2db(mag_spec):
    return 20*np.log10(mag_spec)

%matplotlib inline

# Exercise 1: Building an Edge Detector 2.0

In Lab 2 Exercise 4, we built a simple edge detector by applying a high-pass filter along the rows and columns of an image, then combined the two results to create an image of detected edges. In this exercise, we will build a more sophisticated edge detector by adding onto our original design. This improved detector is known as the [Sobel operator](https://en.wikipedia.org/wiki/Sobel_operator).

The Sobel operator uses the same intuition of finding horizontal and vertical edges separately, then combining these results to form the image. Let $G_x$ be our resulting image from detecting vertical edges (through the x-axis) and $G_y$ be our resulting image from detecting horizontal edges (through the y-axis, then our final result will be given by

$$
G[i,j] = \sqrt{G^2_x[i,j]+G^2_y[i,j]},
$$

where $G[i,j]$ is the pixel value at row $i$, column $j$. We compute $G_x$ and $G_y$ via convolution as follows:

$$
G_x = I * \begin{bmatrix}
1 & 0 & -1\\
\end{bmatrix} * \begin{bmatrix}
1\\
2\\
1\\
\end{bmatrix}
$$

$$
G_y = I * \begin{bmatrix}
1\\
0\\
-1\\
\end{bmatrix} * \begin{bmatrix}
1 & 2 & 1\\
\end{bmatrix},
$$

where $I$ is our original image and $*$ denotes the convolution operator. Note that in the computation of $G_x$ we convolve along the rows with a high-pass filter and convolve along the columns with a low-pass filter, while the computation of $G_y$ reverses this relationship with the same filters.

a. Compute the image $G_x$ by performing the two convolutions along the rows and columns, respectively, using $\textrm{signal.convolve()}$. Plot your resulting image in grayscale. Remember that you should apply the high-pass filter $\begin{bmatrix}1 & 0 & -1\end{bmatrix}$ to each **row** and the low-pass filter $\begin{bmatrix}1 & 2 & 1\end{bmatrix}$ to each **column**. Also, be sure to use the "same" mode when using $\textrm{signal.convolve()}$.

b. Compute the image $G_y$ by performing the two convolutions along the rows and columns, respectively, using $\textrm{signal.convolve()}$. Plot your resulting image in grayscale. Remember that you should apply the high-pass filter $\begin{bmatrix}1 & 0 & -1\end{bmatrix}$ to each **column** and the low-pass filter $\begin{bmatrix}1 & 2 & 1\end{bmatrix}$ to each **row**.

c. Create final result image $G$ according to the above formulation. Plot your resulting image in grayscale.

In [ ]:
#load test-image.jpg
image = imread('test-image.jpg')
n_rows,n_cols = image.shape

#Code for part a.
high_pass = np.array([1, 0, -1])
low_pass = np.array([1, 2, 1])
Gx = np.zeros(image.shape)

for i in range(n_rows):
    Gx[i,:] = signal.convolve(image[i,:], high_pass, mode='same')

for i in range(n_cols):
    Gx[:,i] = signal.convolve(Gx[:,i], low_pass, mode='same')

plt.figure(figsize = (6, 6))
plt.imshow(Gx, cmap='gray')
plt.axis('off')
plt.title("Picture for part a.")


#Code for part b:
Gy = np.zeros(image.shape)

for i in range(n_cols):
    Gy[:,i] = signal.convolve(image[:,i], high_pass, mode='same')


for i in range(n_rows):
    Gy[i,:] = signal.convolve(Gy[i,:], low_pass, mode='same')

plt.figure(figsize = (6, 6))
plt.imshow(Gy, cmap='gray')
plt.axis('off')
plt.title("Picture for part b.")

#Code for part c:
G = np.sqrt(Gx**2 + Gy**2)

plt.figure(figsize = (6, 6))
plt.imshow(G, cmap='gray')
plt.axis('off')
plt.title("Picture for part c.")


# Exercise 2: LCCDE, Transfer Function, and Impulse Response

For each of the following Linear Constant Coefficient Difference Equations (LCCDE), determine the transfer function (numerator and denominator coefficients) in order to plot both the pole-zero plot and impulse response of the system for the requested number of points. **Please plot your impulse responses as a stem plot!** For each system, indicate whether it is BIBO stable, marginally stable, or not BIBO stable and briefly explain your choice.

Note: We have provided the $\textrm{pz\_plot}()$ function from Lab 3 to create your pole-zero plots. Refer to Lab 3 for usage of this function. Some other functions of interest you may want to use from Lab 3 will be $\textrm{signal.dimpulse()}$ and $\textrm{signal.tf2zpk()}$.

a. $y_1[n] = \frac{1}{3}x[n]-2x[n-1]+x[n-2]-
2x[n-3]+\frac{1}{3}x[n-4],\quad 0\leq n < 8$

b. $y_2[n] = x[n] + \frac{1}{4}x[n-2] - 2y_2[n-1] + 3y_2[n-2],\quad 0\leq n < 20$

c. $y_3[n] = x[n] - \frac{1}{2}x[n-2] - y_3[n-3], \quad 0\leq n < 20$

In [ ]:
#Code for exercise 2:
b = [1/3, -2, 1, -2, 1/3]
a = [1, 0, 0, 0, 0]
z, p, k = signal.tf2zpk(b, a)
print('Poles for 2(a):', p)
print('Zeros for 2(a):', z)
pz_plot(z, p, 'Pole-Zero Plot for 2(a)')

n,y = signal.dimpulse((b, a, 1), n=8)
h_n = y[0]

plt.figure()
plt.stem(n, h_n)
plt.xlabel('n')
plt.ylabel('Output Signal')
plt.title('Impulse Response of System 2(a)')

b = [1, 0, 1/4]
a = [1, 2, -3]
z, p, k = signal.tf2zpk(b, a)
print('Poles for 2(b):', p)
print('Zeros for 2(b):', z)
pz_plot(z, p, 'Pole-Zero Plot for 2(b)')

n,y = signal.dimpulse((b, a, 1), n=20)
h_n = y[0]

plt.figure()
plt.stem(n, h_n)
plt.xlabel('n')
plt.ylabel('Output Signal')
plt.title('Impulse Response of System 2(b)')

b = [1, 0, -1/2, 0]
a = [1, 0, 0, 1]
z, p, k = signal.tf2zpk(b, a)
print('Poles for 2(c):', p)
print('Zeros for 2(c):', z)
pz_plot(z, p, 'Pole-Zero Plot for 2(c)')

n,y = signal.dimpulse((b, a, 1), n=20)
h_n = y[0]

plt.figure()
plt.stem(n, h_n)
plt.xlabel('n')
plt.ylabel('Output Signal')
plt.title('Impulse Response of System 2(c)')


## Comments here

Comments for 2.a: It is BIBO stable because all poles in unit circle.


Comments for 2.b: It is not stable because there are poles outside unit circle.


Comments for 2.c: It's marginally stable because you there are poles on the unit circle




# Exercise 3: Windows and Spectral Resolution

**Note: please specify 512 points for each FFT you take in this problem!**

a. Plot the magnitude spectrum (not dB scale) of the following signal using $\textrm{np.fft.rfft()}$.

$$
x_1[n] = 0.5\sin\left(0.5\pi n\right) + 0.02\sin\left(0.6\pi n\right), \quad 0\leq n < 80
$$

What is happening to the second sinusoid with the smaller magnitude in the frequency domain? Is it easy to locate this frequency peak?

b. Fill in the function $\textrm{hamming()}$ below which applies a Hamming window to an input signal. Apply this function to $x_1[n]$ and plot the resulting magnitude spectrum. Is it easier to locate the smaller frequency now?

c. Now let's try resolving two close, but equally large frequency peaks in the below signal $x_2[n]$.

$$
x_2[n] = 0.5\sin\left(0.6\pi n\right) + 0.5\sin\left(0.618\pi n\right), \quad 0\leq n < 80
$$

Plot the magnitude spectrum of $x_2[n]$ (not dB scale) before and after applying the $\textrm{hamming()}$ function. Is it easier to differentiate between the two peaks after applying the Hamming window? Why or why not? **Hint: think about the tradeoff between rectangular and Hamming windows!**


In [ ]:
#Code for part 3.a:
x1 = np.array([0.5 * np.sin(0.5 * np.pi * i) + 0.02 * np.sin(0.6 * np.pi * i) for i in range(80)])

real_fft = np.fft.rfft(x1, 512)
w = np.linspace(0, np.pi, len(real_fft))

plt.figure(figsize = (6, 6))
plt.plot(w, np.abs(real_fft))
plt.title('Input x1 magnitude response')
plt.xlabel('omega')
plt.ylabel('Magnitude')
#Code for part 3.b:
def hamming(x):
    #apply a hamming window to the signal
    windowed_signal = x * np.hamming(len(x))
    
    return windowed_signal

x1_windowed = hamming(x1)

real_fft = np.fft.rfft(x1_windowed, 512)
w = np.linspace(0, np.pi, len(real_fft))

plt.figure(figsize = (6, 6))
plt.plot(w, np.abs(real_fft))
plt.title('Input x1 magnitude response after hamming window')
plt.xlabel('omega')
plt.ylabel('Magnitude')

#Code for part 3.c:
x2 = np.array([0.5 * np.sin(0.6 * np.pi * i) + 0.5 * np.sin(0.618 * np.pi * i) for i in range(80)])

real_fft = np.fft.rfft(x2, 512)
w = np.linspace(0, np.pi, len(real_fft))

plt.figure(figsize = (6, 6))
plt.plot(w, np.abs(real_fft))
plt.title('Input x2 magnitude response')
plt.xlabel('omega')
plt.ylabel('Magnitude')
x2_windowed = hamming(x2)

real_fft = np.fft.rfft(x2_windowed, 512)
w = np.linspace(0, np.pi, len(real_fft))

plt.figure(figsize = (6, 6))
plt.plot(w, np.abs(real_fft))
plt.title('Input x2 magnitude response after hamming window')
plt.xlabel('omega')
plt.ylabel('Magnitude')


## Comments here

Comments for part 3.a: Since the peak's magnitude is too small (only 0.02), it is really hard to spot it on the graph, we can barely see a little bit. It is not easy to locate.


Comments for part 3.b: Yes it is easier to locate this smaller frequency now. The Hamming window reduces the sidelobes of the large frequency, so we can see the small frequency sticking out much easier.


Comments for part 3.c: No, it is harder to differentiate between the 2 peaks after the hamming window, this is because hamming window have wider main lobes compared to rectangular window, so it connects the 2 really close peak together.



# Exercise 4: Chirp Redux

For this exercise, we will revisit the chirp activity from Lab 4 but this time with the help of spectrograms to visualize our chirps. The below provided code creates a five second long chirp signal with sampling rate $f_s = 44,100~Hz$ and maximum signal frequency $22,050~Hz$. Recall that the $\textrm{signal.chirp}$ function linearly sweeps between the requested frequencies over the set time interval. Hint: referring back to Labs 4 and 5 will help with understanding the chirp function and plotting spectrograms, respectively.

a. Plot the spectrogram of the original chirp signal we have generated for you.

b. Reduce the sampling rate of the original chirp signal by a factor of 9. Plot the resulting spectrogram and explain what you see. If we listened to this audio signal, how many rises and falls would you hear? **Note: you may either create a new chirp signal using the requested lower sampling or perform downsampling without an anti-aliasing filter.**

In [ ]:
Fs = 44100 #sampling rate for audio clip in Hz
t1 = 5 #make clips 5 seconds
t = np.linspace(0,t1,t1*Fs)
f0 = 0 #start frequency (Hz)
f1 = 22050 #end frequency (Hz)
chirp_original = signal.chirp(t, f0 = f0, t1 = t1, f1 = f1)
nfft = 1024

#Code for part 4.a:
f_a, t_a, S_a = signal.spectrogram(chirp_original, Fs, nperseg = nfft, noverlap = int(nfft/2), nfft = nfft)

plt.figure(figsize=(6,6))
plt.pcolormesh(t_a, f_a, sig2db(S_a))
plt.title('Spectrogram for original chirp')
plt.ylim([0, 22050])
plt.ylabel('Frequency [Hz]')
plt.xlabel('Time [sec]')
plt.colorbar()
#Code for part 4.b:
current_fs = 4900 # sampling rate for audio clip in Hz
t1 = 5 # make clips 5 seconds
t = np.linspace(0, t1, t1*current_fs) # make sure to specify the number of points to match desired sampling frequency!!!
f0 = 0 # start frequency (Hz)
f1 = 22050 # end frequency (Hz)
chirps = signal.chirp(t, f0=f0, t1=t1, f1=f1)

f_a, t_a, S_a = signal.spectrogram(chirps, current_fs, nperseg = nfft, noverlap = int(nfft/2), nfft = nfft)

plt.figure(figsize=(6,6))
plt.pcolormesh(t_a, f_a, sig2db(S_a))
plt.title('Spectrogram for reduced sampled chirp')
plt.ylim([0, 22050/9])
plt.ylabel('Frequency [Hz]')
plt.xlabel('Time [sec]')
plt.colorbar()
Audio(data = chirps, rate = current_fs)


## Comments here

Comments for 4.b: We see the frequency of the sound goes up and down 9 times ranging from 0 to 2450 Hz. This is because we reduced the sample rate to 4900 Hz meaning anything over Nyquist rate 2450 Hz can't be restored. It rises 5 times and falls down 4 times



# Exercise 5: FIR Filter Design

a. Given the audio signal file ``Sound_original.wav``, compute and display the full FFT magnitude spectrum (no dB scale). Try listening to it!

b. Now let's assume that we pass this audio signal into a system described as followed.

<img src="./sys_illus.png" alt="Drawing" style="width: 600px;"/>

where $x[n]$ is our audio signal, $H(\omega)$ is an LTI system with impulse response $h[n]$, $d[n]$ is some noise, and $y[n]$ is the ouput. 
For this question, we want to simulate the ouput result $y[n]$.

We only know that the LTI system $H(\omega)$ is acting like a low-pass filter with a cutoff frequency of $\frac{\pi}{3}$. Use $\textrm{signal.remez()}$ to obtain the impulse response of this system assuming the filter length is $N = 100$ and the transition bandwidth is $\frac{\pi}{10}$. Plot magnitude response **on a dB scale** using the provided $\textrm{sig2db()}$ function. 

c. Now, we can obtain output $y[n]$ by filtering $x[n]$ with $h[n]$ and adding $d[n]$ after filtering. 

Here, $d[n]$ is assumed to be additive white Gaussian noise (AWGN). We can create our noise $d[n]$ by typing

``d = 2500 * np.random.randn(len(x_filtered))``

Compute $y[n]$ by summing your filtered audio signal $x[n]*h[n]$ and d. Try listening to y! Plot the FFT magnitude spectrum of $d[n]$ and the FFT magnitude spectrum of $y[n]$ on separate figures (no dB scale). Judging from these two graphs, do you think the simple filtering methods we have discussed in class will be able to perfectly separate our noise $d[n]$ from the filtered audio $x[n]*h[n]$?

In [ ]:
fs,original = wavfile.read('Sound_original.wav')


#Code for 5.a:
fft = np.fft.fft(original)
fft = np.fft.fftshift(fft)
w = np.linspace(-np.pi, np.pi, len(fft))

plt.figure(figsize=(6,6))
plt.plot(w, np.abs(fft))
plt.title('Full FFT magnitude spectrum')
plt.xlabel('omega')
plt.ylabel('Magnitude')


#Code for 5.b:
lpf_bands = [0, 1/3 - 0.05, 1/3+0.05, 1]
lpf_desired = [1, 0]
lpf = signal.remez(100, lpf_bands, lpf_desired, fs=2)
w, H_lpf = signal.freqz(lpf, 1, 512)

plt.figure(figsize=(6,6))
plt.plot(w, sig2db(np.abs(H_lpf)))
plt.title('Frequency response of low-pass filter')
plt.xlabel('Frequency')
plt.ylabel('Magnitude (db)')

#Code for 5.c:
x_filtered = signal.convolve(original, lpf, mode = "same")
d = 2500 * np.random.randn(len(x_filtered))
y = x_filtered + d

fft = np.fft.fft(y)
fft = np.fft.fftshift(fft)
w = np.linspace(-np.pi, np.pi, len(fft))

plt.figure(figsize=(6,6))
plt.plot(w, np.abs(fft))
plt.title('Frequency response of y[n]')
plt.xlabel('Frequency')
plt.ylabel('Magnitude')

fft = np.fft.fft(d)
fft = np.fft.fftshift(fft)
w = np.linspace(-np.pi, np.pi, len(fft))

plt.figure(figsize=(6,6))
plt.plot(w, np.abs(fft))
plt.title('Frequency response of d[n]')
plt.xlabel('Frequency')
plt.ylabel('Magnitude')

Audio(data = y, rate = fs)

## Comments here

Comments for 5.c: No, I don't think the filtering methods we have discussed in class will be able to separate our noise since our filters are designed to get rid of a certain frequency, but in this case we have noise across all the frequencies, including the ones with the actual audio.



# Submission Instructions:

Please place all files in one folder, compress that folder as a zip file, and name the zip file ``netid_labfinal``. Submit your zip file to Canvas like previous labs.